In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
num = 4
device = qml.device('default.qubit',wires=4)
weightage_zz = [1.0]*(num-1)
weightage_x = [1.0]*(num)
observable_zz = [qml.PauliZ(i)@qml.PauliZ(i+1) for i in range(num-1)]
observable_x = [qml.PauliX(i) for i in range(num)]
Hamil_zz = qml.Hamiltonian(weightage_zz,observable_zz)
Hamil_x = qml.Hamiltonian(weightage_x,observable_x)

In [5]:
Hamil = Hamil_zz+Hamil_x
trotterization = 6
@qml.qnode(device)
def quantum_circuit(params):
    for w in range(num):
        qml.Hadamard(w)
    for i in range(len(params)):
        qml.ApproxTimeEvolution(Hamil_zz,params[0][i],6)
        qml.ApproxTimeEvolution(Hamil_x,params[1][i],6)
    return qml.expval(Hamil)

In [7]:
np.random.seed(42)
length_of_parameter = 12
params = np.random.uniform(0,0.1,(2,length_of_parameter),requires_grad = True)
optimiser = qml.GradientDescentOptimizer(stepsize = 0.3)
steps = 50
for i in range(steps):
    params,cost = optimiser.step_and_cost(quantum_circuit,params)
    if(i%9==0):
        print(f'current paramter is {params} and cost is {cost}')
        print('\n')
print(f'final paramter is {params}')
print(f'final cost is {quantum_circuit(params)}')

current paramter is [[ 0.21161161  0.86491799  0.07319939  0.05986585  0.01560186  0.01559945
   0.00580836  0.08661761  0.0601115   0.07080726  0.00205845  0.09699099]
 [-0.19572246 -0.87534888  0.0181825   0.01834045  0.03042422  0.05247564
   0.0431945   0.02912291  0.06118529  0.01394939  0.02921446  0.03663618]] and cost is 3.9346202541598663


current paramter is [[-0.04591177  1.75994529  0.07319939  0.05986585  0.01560186  0.01559945
   0.00580836  0.08661761  0.0601115   0.07080726  0.00205845  0.09699099]
 [-0.15920425 -0.71065294  0.0181825   0.01834045  0.03042422  0.05247564
   0.0431945   0.02912291  0.06118529  0.01394939  0.02921446  0.03663618]] and cost is -0.3213603746605979


current paramter is [[-4.24011513e+00 -2.19810122e+00  7.31993942e-02  5.98658484e-02
   1.56018640e-02  1.55994520e-02  5.80836122e-03  8.66176146e-02
   6.01115012e-02  7.08072578e-02  2.05844943e-03  9.69909852e-02]
 [-4.99371812e+00 -6.78390068e-01  1.81824967e-02  1.83404510e-02
   3.04242